# Visualization & Dashboard

**Continuing from:** `feature_engineering_analysis.ipynb` — that notebook turned the clean Superstore data into a set of analysis-ready tables and answered the five business questions in text/table form. This notebook's job is purely presentational: turn those already-computed tables into charts, without recomputing any aggregation logic.

> **Note on notebooks and kernels:** this notebook does **not** share memory with `feature_engineering_analysis.ipynb` — each `.ipynb` file runs its own kernel. Step 0 below loads the tables that notebook exported to `handoff_data/`, rather than assuming they already exist in memory.

### Step 0 — Setup & Recap

Load every table named in the feature-engineering notebook's `handoff_manifest`, straight back from `handoff_data/`. Nothing is recomputed here — this is a read-only reload of work already done.

In [1]:
# Importing the proper and required data + visualization packages
import pandas as pd
import matplotlib.pyplot as plt
import pyarrow

In [2]:
handoff_dir = "handoff_data"

# Row-level / per-entity tables
df_clean = pd.read_parquet(f"{handoff_dir}/df_clean.parquet", engine="pyarrow")
customer_features = pd.read_parquet(f"{handoff_dir}/customer_features.parquet", engine="pyarrow")
product_features = pd.read_parquet(f"{handoff_dir}/product_features.parquet", engine="pyarrow")
rfm = pd.read_parquet(f"{handoff_dir}/rfm.parquet", engine="pyarrow")

# Small, chart-ready summary tables
state_profit = pd.read_csv(f"{handoff_dir}/state_profit.csv", index_col="State").squeeze("columns")
discount_profit = pd.read_csv(f"{handoff_dir}/discount_profit.csv", index_col="Discount Bucket").squeeze("columns")
ship_mode_summary = pd.read_csv(f"{handoff_dir}/ship_mode_summary.csv", index_col="Ship Mode")
region_summary = pd.read_csv(f"{handoff_dir}/region_summary.csv", index_col="Region")
category_summary = pd.read_csv(f"{handoff_dir}/category_summary.csv", index_col="Category")
pivot = pd.read_csv(f"{handoff_dir}/pivot_region_category.csv", index_col="Region")

df_clean.shape

(9993, 29)

In [3]:
# Confirm every table survived the handoff intact
df_clean.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 9993 entries, 0 to 9992
Data columns (total 29 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   Order ID         9993 non-null   str           
 1   Order Date       9993 non-null   datetime64[us]
 2   Ship Date        9993 non-null   datetime64[us]
 3   Ship Mode        9993 non-null   category      
 4   Customer ID      9993 non-null   str           
 5   Customer Name    9993 non-null   str           
 6   Segment          9993 non-null   category      
 7   Country/Region   9993 non-null   category      
 8   City             9993 non-null   str           
 9   State            9993 non-null   category      
 10  Postal Code      9993 non-null   str           
 11  Region           9993 non-null   category      
 12  Product ID       9993 non-null   str           
 13  Category         9993 non-null   category      
 14  Sub-Category     9993 non-null   category      
 15

### Step 1 — Charts

Build one chart per question from the feature-engineering notebook's Step 6, using the corresponding table loaded above:

1. `state_profit` → most/least profitable states
2. `discount_profit` → Discount vs. Profit
3. `ship_mode_summary` → Ship Mode vs. profitability / shipping time
4. `df_clean` (`Order Month` / `Order Weekday`) → seasonality in Sales
5. `rfm` → most valuable customers